# Questão 5 — Análise de Clientes

## Objetivo
Identificar os clientes fiéis da LH Nautical, considerando não apenas alto gasto, mas também recorrência e diversidade de categorias consumidas.

## Regras de negócio aplicadas
- **Faturamento Total**: soma do valor vendido por cliente
- **Frequência**: contagem de transações distintas por cliente
- **Ticket Médio**: faturamento total dividido pela frequência
- **Diversidade de Categorias**: quantidade de categorias distintas consumidas por cliente
- **Filtro de Elite**: apenas clientes com compras em **3 ou mais categorias**
- **Desempate**: em caso de empate no ticket médio, ordenar por `customer_id` crescente

## Estratégia adotada
A análise foi construída sobre o modelo dimensional já tratado no dbt:
- `fct_vendas`: fato principal com as transações
- `dim_produto`: dimensão com categoria padronizada
- `mart_clientes_fieis`: ranking final dos 10 clientes selecionados
- `mart_categoria_top10_clientes`: categoria mais consumida por esse grupo

In [0]:
%sql
-- Conferência inicial da fato de vendas
SELECT *
FROM lh_nautical.04_gold.fct_vendas
LIMIT 10;

sale_id,customer_id,product_id,sale_date,quantity,receita_transacao_brl,custo_unitario_brl,custo_total_brl,prejuizo_brl,teve_prejuizo,cost_start_date,exchange_rate_date,exchange_rate
0,42,105,2023-09-10,11,3405.00,312.17,3433.83,28.83,true,2023-04-17,2023-09-08,4.983500
1,3,136,2024-09-15,9,16873.90,2176.42,19587.76,2713.86,true,2023-10-20,2024-09-13,5.571700
2,25,139,2024-08-13,7,9475.30,1354.81,9483.66,8.36,true,2020-10-30,2024-08-13,5.487500
4,20,23,2023-02-03,5,55893.00,10760.90,53804.50,0.00,false,2022-10-13,2023-02-03,5.103000
5,8,57,2024-02-12,4,451403.90,111852.26,447409.05,0.00,false,2022-12-16,2024-02-09,4.971700
6,36,52,2023-09-26,3,39056.40,13098.89,39296.66,240.26,true,2023-01-11,2023-09-26,4.971700
8,27,25,2024-02-28,3,34560.05,12282.60,36847.81,2287.76,true,2023-11-29,2024-02-28,4.955700
9,37,26,2023-11-07,7,114932.90,16610.49,116273.41,1340.51,true,2022-03-14,2023-11-07,4.867000
10,31,143,2024-08-25,3,12643.55,4956.15,14868.45,2224.90,true,2023-05-24,2024-08-23,5.526300
11,39,128,2023-05-07,5,23254.00,4239.52,21197.58,0.00,false,2022-07-22,2023-05-05,4.969600


In [0]:
%sql
-- Conferência inicial da dimensão de produtos
-- coluna "category" representa a categoria já padronizada para análise.
-- coluna "actual_category" preserva a grafia original para auditoria.
SELECT
    product_id,
    product_name,
    category,
    actual_category
FROM lh_nautical.04_gold.dim_produto
LIMIT 20;

product_id,product_name,category,actual_category
1,Transponder AIS Maré Magnum,eletronicos,ELETRONICOS
2,Transponder Furuno Marlin,eletronicos,ELETRONICOS
3,Radar Furuno Pulse Leviathan,eletronicos,E L E T R Ô N I C O S
4,Rádio AIS Hydro Tidal Zen,eletronicos,Eletrunicos
5,Piloto Automático Furuno Storm,eletronicos,Eletronicoz
6,Transponder AIS Vector,eletronicos,Eletrunicos
7,Radar AIS Zen,eletronicos,eLeTrÔnIcOs
8,GPS AIS Zen,eletronicos,E L E T R Ô N I C O S
9,Transponder AIS Titan Pulse,eletronicos,Eletronicoz
10,Piloto Automático Simrad Titan Flux Magnum,eletronicos,eletrônicos


## Etapa 2 — Cálculo dos indicadores por cliente

A consulta abaixo demonstra a lógica de negócio aplicada para:
- calcular faturamento total
- calcular frequência
- calcular ticket médio
- medir diversidade de categorias
- aplicar o filtro de elite
- selecionar os Top 10 clientes fiéis

In [0]:
%sql
-- Questão 5.1
-- Calcula:
-- 1) Ticket médio e diversidade de categorias por cliente
-- 2) Top 10 clientes fiéis (maior ticket médio entre clientes com diversidade >= 3)
-- 3) Categoria mais vendida em quantidade total de itens para esse grupo

WITH base_vendas AS (

    SELECT
        f.sale_id,
        f.customer_id,
        f.product_id,
        f.quantity,
        f.receita_transacao_brl,
        p.category
    FROM lh_nautical.04_gold.fct_vendas f
    LEFT JOIN lh_nautical.04_gold.dim_produto p
        ON f.product_id = p.product_id

),

indicadores_cliente AS (

    SELECT
        customer_id,

        -- Faturamento total do cliente
        SUM(receita_transacao_brl) AS faturamento_total,

        -- Frequência de compras
        COUNT(DISTINCT sale_id) AS frequencia,

        -- Ticket médio
        ROUND(SUM(receita_transacao_brl) / COUNT(DISTINCT sale_id), 2) AS ticket_medio,

        -- Diversidade de categorias compradas
        COUNT(DISTINCT category) AS diversidade_categorias

    FROM base_vendas
    GROUP BY customer_id

),

top_10_clientes_fieis AS (

    SELECT
        customer_id,
        faturamento_total,
        frequencia,
        ticket_medio,
        diversidade_categorias
    FROM indicadores_cliente
    WHERE diversidade_categorias >= 3
    ORDER BY ticket_medio DESC, customer_id ASC
    LIMIT 10

),

categoria_top_10 AS (

    SELECT
        b.category,
        SUM(b.quantity) AS quantidade_total_itens
    FROM base_vendas b
    INNER JOIN top_10_clientes_fieis t
        ON b.customer_id = t.customer_id
    GROUP BY b.category
    ORDER BY quantidade_total_itens DESC, category ASC
    LIMIT 1

)

SELECT
    t.customer_id,
    t.faturamento_total,
    t.frequencia,
    t.ticket_medio,
    t.diversidade_categorias,
    c.category AS categoria_mais_vendida_top_10,
    c.quantidade_total_itens AS qtd_itens_categoria_top_10
FROM top_10_clientes_fieis t
CROSS JOIN categoria_top_10 c
ORDER BY t.ticket_medio DESC, t.customer_id ASC;

customer_id,faturamento_total,frequencia,ticket_medio,diversidade_categorias,categoria_mais_vendida_top_10,qtd_itens_categoria_top_10
47,64003343.75,190,336859.70,3,propulsao,6030
42,72187369.50,222,325168.33,3,propulsao,6030
9,66788855.35,218,306370.90,3,propulsao,6030
22,59581398.75,198,300916.16,3,propulsao,6030
2,65652931.35,220,298422.42,3,propulsao,6030
28,60826837.25,204,298170.77,3,propulsao,6030
46,59126834.35,199,297119.77,3,propulsao,6030
38,57093331.15,195,292786.31,3,propulsao,6030
36,62791038.15,215,292051.34,3,propulsao,6030
5,58592802.70,202,290063.38,3,propulsao,6030


In [0]:
# Consulta os dados das marts já criadas
df_top10 = spark.sql("""
SELECT *
FROM lh_nautical.04_gold.mart_clientes_fieis
ORDER BY ticket_medio DESC, customer_id ASC
""")

df_categoria = spark.sql("""
SELECT *
FROM lh_nautical.04_gold.mart_categoria_top10_clientes
ORDER BY quantidade_total_itens DESC, category ASC
LIMIT 1
""")

# Pega os valores principais
top_1 = df_top10.collect()[0]
categoria_top = df_categoria.collect()[0]

# Prints formatados
print("RESULTADO DA QUESTÃO 5\n")

print(f"Quantidade de clientes fiéis selecionados: {df_top10.count()}")

print(f"\nCliente com maior Ticket Médio: {top_1['customer_id']}")
print(f"Ticket Médio: {round(top_1['ticket_medio'], 2)}")
print(f"Diversidade de categorias: {top_1['diversidade_categorias']}")

print(f"\nCategoria mais vendida para os Top 10 clientes: {categoria_top['category']}")
print(f"Quantidade total de itens: {categoria_top['quantidade_total_itens']}")

RESULTADO DA QUESTÃO 5

Quantidade de clientes fiéis selecionados: 10

Cliente com maior Ticket Médio: 47
Ticket Médio: 336859.70
Diversidade de categorias: 3

Categoria mais vendida para os Top 10 clientes: propulsao
Quantidade total de itens: 6030


## Etapa 3 — Uso da mart final de clientes fiéis

Como parte da modelagem analítica, a regra acima foi materializada em uma mart final.
Isso facilita a rastreabilidade e reaproveitamento da análise.

In [0]:
%sql
-- Resultado da seleção dos Top 10 clientes fiéis
SELECT *
FROM lh_nautical.04_gold.mart_clientes_fieis
ORDER BY ticket_medio DESC, customer_id ASC;

customer_id,faturamento_total,frequencia,ticket_medio,diversidade_categorias
47,64003343.75,190,336859.703947368421,3
42,72187369.50,222,325168.331081081081,3
9,66788855.35,218,306370.896100917431,3
22,59581398.75,198,300916.155303030303,3
2,65652931.35,220,298422.415227272727,3
28,60826837.25,204,298170.770833333333,3
46,59126834.35,199,297119.770603015075,3
38,57093331.15,195,292786.313589743590,3
36,62791038.15,215,292051.340232558140,3
5,58592802.70,202,290063.379702970297,3


## Etapa 4 — Categoria mais comprada pelos Top 10 clientes

Após identificar os clientes fiéis, o próximo passo é descobrir
qual categoria concentrou a maior quantidade total de itens comprados por esse grupo.

In [0]:
%sql
-- Questão 5.1 (parte final)
-- Categoria mais vendida, em quantidade de itens, considerando apenas os Top 10 clientes selecionados

WITH top_10 AS (

    SELECT
        customer_id
    FROM lh_nautical.04_gold.mart_clientes_fieis

),

base AS (

    SELECT
        f.customer_id,
        f.quantity,
        p.category
    FROM lh_nautical.04_gold.fct_vendas f
    LEFT JOIN lh_nautical.04_gold.dim_produto p
        ON f.product_id = p.product_id
    INNER JOIN top_10 t
        ON f.customer_id = t.customer_id

),

categoria_agregada AS (

    SELECT
        category,
        SUM(quantity) AS quantidade_total_itens
    FROM base
    GROUP BY category

)

SELECT *
FROM categoria_agregada
ORDER BY quantidade_total_itens DESC, category ASC;

category,quantidade_total_itens
propulsao,6030
ancoragem,5632
eletronicos,5214


## Questão 5.3 — Explicação da lógica aplicada

### 1. Como foi realizada a limpeza das categorias?
A limpeza das categorias foi realizada anteriormente na preparação da dimensão de produtos.  
Os nomes inconsistentes e com erro de grafia foram consolidados em uma categoria padronizada, preservando a coluna original para rastreabilidade.  
Na análise final, foi utilizada a coluna `category`, já padronizada, garantindo que variações como grafias incorretas não gerassem duplicidade analítica.

### 2. Qual lógica foi utilizada para filtrar os clientes com diversidade mínima?
Para cada cliente, foi calculada a quantidade de categorias distintas compradas por meio de `COUNT(DISTINCT category)`.  
Em seguida, foi aplicado o critério de elite definido pela regra de negócio: somente clientes com diversidade de categorias maior ou igual a 3 foram mantidos no ranking.

### 3. Como foi garantido que a contagem de itens refletisse apenas os Top 10?
Primeiro foi gerada a lista final dos 10 clientes fiéis com base no maior ticket médio, respeitando o filtro de diversidade mínima.

Depois, ao calcular a categoria mais vendida, a análise foi restrita exclusivamente a esse conjunto por meio de um `INNER JOIN` entre a fato de vendas e a mart dos Top 10 clientes.  
Dessa forma, a soma de `quantity` considerou apenas compras realizadas pelos clientes efetivamente selecionados.